# Chapter 2.7: Disaggregated Serving -- Prefill/Decode Separation

## Learning Objectives

By the end of this notebook, you will:

1. **Understand why prefill and decode have fundamentally different compute profiles**
2. **Explain the benefits of disaggregation**: independent scaling, better GPU utilization
3. **Analyze KV-cache transfer** between prefill and decode nodes
4. **Understand network bandwidth requirements** for disaggregated architectures
5. **Compare DistServe, Splitwise, and Mooncake** architecture designs
6. **Simulate disaggregated vs co-located serving** and compare performance
7. **Design a disaggregated architecture** for a given workload

## Prerequisites

- Chapter 2.1: Request Lifecycle (prefill vs decode distinction)
- Chapter 2.3: KV-Cache Memory Management
- Chapter 2.4: Model Parallelism Strategies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part2/chapter_2.7_disaggregated.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part2/chapter_2.7_disaggregated.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple
from enum import Enum
import time
import warnings
warnings.filterwarnings('ignore')

## 1. Why Disaggregate? The Prefill-Decode Mismatch

Recall from Chapter 2.1 that **prefill** and **decode** have fundamentally different compute characteristics:

| Property | Prefill | Decode |
|----------|---------|--------|
| Compute pattern | Matrix-Matrix (GEMM) | Matrix-Vector (GEMV) |
| Bottleneck | **Compute-bound** | **Memory-bandwidth-bound** |
| GPU utilization | **High** (80-95%) | **Low** (10-30%) |
| Batch size sweet spot | Small (1-4) | Large (32-256) |
| Latency sensitivity | Moderate (affects TTFT) | Critical (affects TPOT) |
| KV-cache | Generates it | Reads from it |
| Memory usage pattern | Burst (allocate all at once) | Incremental (1 token/step) |

### The Problem with Co-Located Serving

When prefill and decode share the same GPUs (co-located):

1. **Prefill starves decode**: A long prefill blocks all decode operations, spiking TPOT
2. **Decode wastes compute**: Decode barely uses GPU compute, leaving FLOPS idle
3. **Cannot scale independently**: More prefill requests need different resources than more decode tokens
4. **Memory contention**: Prefill needs burst allocation, decode needs stable long-term allocation

In [ ]:
## Figure: Disaggregated Serving -- Prefill and Decode Separation
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(18, 8))
ax.set_xlim(0, 20)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Disaggregated Architecture: Separate Prefill and Decode Nodes', fontsize=16, fontweight='bold')

BLUE, GREEN, ORANGE, RED, PURPLE = '#4A90D9', '#27AE60', '#F39C12', '#E74C3C', '#8E44AD'

# Prefill Nodes (left)
ax.text(4, 9.5, 'PREFILL NODES', fontsize=14, fontweight='bold', color=RED, ha='center')
ax.text(4, 9.0, '(Compute-optimized)', fontsize=10, color=RED, ha='center', style='italic')
for i in range(3):
    y = 7.5 - i * 2
    ax.add_patch(patches.FancyBboxPatch((2, y), 4, 1.5, boxstyle="round,pad=0.15",
                 facecolor=RED, alpha=0.7, edgecolor='black', linewidth=1.5))
    ax.text(4, y + 0.75, f'Prefill GPU {i+1}\nHigh compute util (~90%)', ha='center', va='center',
            fontsize=9, fontweight='bold', color='white')

# KV Transfer (middle)
ax.add_patch(patches.FancyBboxPatch((8, 4), 4, 3, boxstyle="round,pad=0.2",
             facecolor=GREEN, alpha=0.5, edgecolor='black', linewidth=2))
ax.text(10, 6.5, 'KV-Cache\nTransfer', ha='center', fontsize=12, fontweight='bold', color='white')
ax.text(10, 5.5, 'via RDMA /\nInfiniBand', ha='center', fontsize=9, color='white')
ax.text(10, 4.5, '~2-10 ms\nper request', ha='center', fontsize=9, color='white', style='italic')

# Arrows: Prefill -> KV Transfer
for i in range(3):
    y = 8.25 - i * 2
    ax.annotate('', xy=(8, 5.5), xytext=(6, y),
                arrowprops=dict(arrowstyle='->', lw=2, color=GREEN, alpha=0.6))

# Decode Nodes (right)
ax.text(16, 9.5, 'DECODE NODES', fontsize=14, fontweight='bold', color=BLUE, ha='center')
ax.text(16, 9.0, '(Memory BW-optimized)', fontsize=10, color=BLUE, ha='center', style='italic')
for i in range(4):
    y = 7.8 - i * 1.8
    ax.add_patch(patches.FancyBboxPatch((14, y), 4, 1.3, boxstyle="round,pad=0.15",
                 facecolor=BLUE, alpha=0.7, edgecolor='black', linewidth=1.5))
    ax.text(16, y + 0.65, f'Decode GPU {i+1}\nLarge batch, high BW util', ha='center', va='center',
            fontsize=8, fontweight='bold', color='white')

# Arrows: KV Transfer -> Decode
for i in range(4):
    y = 8.45 - i * 1.8
    ax.annotate('', xy=(14, y), xytext=(12, 5.5),
                arrowprops=dict(arrowstyle='->', lw=2, color=GREEN, alpha=0.6))

# Benefits box at bottom
benefits = [
    'Independent scaling (add prefill or decode GPUs separately)',
    'No prefill-decode interference (decode latency stays stable)',
    'Optimized batch sizes per phase',
]
ax.text(10, 1.5, 'Benefits of Disaggregation:', fontsize=11, fontweight='bold', ha='center')
for i, b in enumerate(benefits):
    ax.text(10, 0.9 - i*0.4, f'{i+1}. {b}', fontsize=9, ha='center')

plt.tight_layout()
plt.show()
print("Prefill nodes are compute-bound -> optimize for FLOPS.")
print("Decode nodes are memory-bound -> optimize for bandwidth and large batches.")
print("The critical path is KV-cache transfer between nodes.")

In [ ]:
# Visualize the compute profile mismatch

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: GPU Utilization during prefill vs decode
ax = axes[0, 0]
operations = ['Prefill\n(512 tokens)', 'Decode\n(batch=1)', 'Decode\n(batch=32)', 'Decode\n(batch=128)']
gpu_util = [92, 8, 25, 55]
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
bars = ax.bar(operations, gpu_util, color=colors, alpha=0.8, edgecolor='black')
ax.set_ylabel('GPU Compute Utilization (%)')
ax.set_title('GPU Utilization: Prefill vs Decode', fontweight='bold')
ax.axhline(y=50, color='red', linestyle='--', alpha=0.5, label='50% threshold')
for bar, val in zip(bars, gpu_util):
    ax.text(bar.get_x() + bar.get_width()/2, val + 2, f'{val}%', ha='center', fontweight='bold')
ax.set_ylim(0, 105)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Plot 2: Memory bandwidth utilization
ax = axes[0, 1]
mem_bw_util = [30, 85, 90, 95]
bars = ax.bar(operations, mem_bw_util, color=colors, alpha=0.8, edgecolor='black')
ax.set_ylabel('Memory Bandwidth Utilization (%)')
ax.set_title('Memory BW Utilization: Prefill vs Decode', fontweight='bold')
for bar, val in zip(bars, mem_bw_util):
    ax.text(bar.get_x() + bar.get_width()/2, val + 2, f'{val}%', ha='center', fontweight='bold')
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)

# Plot 3: Arithmetic intensity
ax = axes[1, 0]
seq_lens = np.arange(1, 4097)
hidden_dim = 4096
# Arithmetic intensity = FLOPs / Memory_access
# Prefill: proportional to seq_len (GEMM)
# Decode: approximately constant (GEMV)
prefill_ai = 2 * seq_lens  # Simplified
decode_ai = np.full_like(seq_lens, dtype=float, fill_value=2.0)

ax.plot(seq_lens, prefill_ai, color='#e74c3c', linewidth=2, label='Prefill')
ax.plot(seq_lens, decode_ai, color='#3498db', linewidth=2, label='Decode (per token)')

# GPU roofline threshold
ax.axhline(y=160, color='gray', linestyle='--', alpha=0.5, label='A100 Compute-Memory Boundary')
ax.fill_between(seq_lens, 0, 160, alpha=0.1, color='#3498db', label='Memory-bound region')
ax.fill_between(seq_lens, 160, prefill_ai, where=prefill_ai > 160, alpha=0.1, color='#e74c3c', label='Compute-bound region')

ax.set_xlabel('Sequence Length (tokens)')
ax.set_ylabel('Arithmetic Intensity (FLOPs/byte)')
ax.set_title('Arithmetic Intensity: Prefill vs Decode', fontweight='bold')
ax.legend(fontsize=8)
ax.set_yscale('log')
ax.set_xscale('log')
ax.grid(alpha=0.3)

# Plot 4: Optimal batch size
ax = axes[1, 1]
batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256]
prefill_throughput = [100, 180, 320, 500, 650, 700, 720, 730, 735]  # saturates quickly
decode_throughput = [50, 95, 180, 340, 600, 1050, 1600, 2100, 2400]  # keeps scaling

ax.plot(batch_sizes, prefill_throughput, 'o-', color='#e74c3c', linewidth=2, label='Prefill (tokens/s)')
ax.plot(batch_sizes, decode_throughput, 's-', color='#3498db', linewidth=2, label='Decode (tokens/s)')
ax.set_xlabel('Batch Size')
ax.set_ylabel('Throughput (tokens/sec)')
ax.set_title('Throughput vs Batch Size', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xscale('log', base=2)

plt.suptitle('The Prefill-Decode Mismatch: Why Disaggregate?', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 2. Disaggregated Architecture Overview

In a disaggregated serving architecture:

- **Prefill nodes**: Dedicated GPUs that only run prefill. Optimized for compute throughput.
- **Decode nodes**: Dedicated GPUs that only run decode. Optimized for memory bandwidth.
- **KV-cache transfer**: After prefill, the KV-cache is sent from prefill node to decode node.
- **Router**: Directs incoming requests to prefill nodes, then assigns to decode nodes.

### The Key Insight

By separating prefill and decode, each can be independently optimized:
- Prefill nodes can use smaller batch sizes for lower TTFT
- Decode nodes can use larger batch sizes for higher throughput
- Each type can be scaled independently based on workload

In [ ]:
# Visualize disaggregated architecture

fig, ax = plt.subplots(figsize=(18, 10))
ax.set_xlim(0, 18)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('Disaggregated LLM Serving Architecture', fontsize=16, fontweight='bold', pad=20)

# Client
ax.add_patch(FancyBboxPatch((0.5, 5), 2, 2, boxstyle="round,pad=0.2",
                             facecolor='#95a5a6', alpha=0.7, edgecolor='black', linewidth=2))
ax.text(1.5, 6, 'Clients', ha='center', va='center', fontsize=11, fontweight='bold')

# Router/Load Balancer
ax.add_patch(FancyBboxPatch((3.5, 4.5), 2.5, 3, boxstyle="round,pad=0.2",
                             facecolor='#f39c12', alpha=0.7, edgecolor='black', linewidth=2))
ax.text(4.75, 6, 'Router /\nLoad Balancer', ha='center', va='center', fontsize=10, fontweight='bold')
ax.text(4.75, 5, 'Schedule\nPrefill->Decode', ha='center', va='center', fontsize=8, style='italic')

# Prefill Nodes
for i in range(3):
    y = 9 - i * 2.5
    ax.add_patch(FancyBboxPatch((7.5, y), 2.5, 1.5, boxstyle="round,pad=0.15",
                                 facecolor='#e74c3c', alpha=0.7, edgecolor='black', linewidth=1.5))
    ax.text(8.75, y + 0.75, f'Prefill Node {i+1}\n(Compute-optimized)', 
            ha='center', va='center', fontsize=8, fontweight='bold', color='white')

ax.text(8.75, 11, 'Prefill Pool', ha='center', fontsize=12, fontweight='bold', color='#e74c3c')

# Decode Nodes
for i in range(4):
    y = 9.5 - i * 2
    ax.add_patch(FancyBboxPatch((13, y), 2.5, 1.5, boxstyle="round,pad=0.15",
                                 facecolor='#3498db', alpha=0.7, edgecolor='black', linewidth=1.5))
    ax.text(14.25, y + 0.75, f'Decode Node {i+1}\n(Memory-optimized)', 
            ha='center', va='center', fontsize=8, fontweight='bold', color='white')

ax.text(14.25, 11.5, 'Decode Pool', ha='center', fontsize=12, fontweight='bold', color='#3498db')

# Arrows
# Client -> Router
ax.annotate('', xy=(3.5, 6), xytext=(2.5, 6), arrowprops=dict(arrowstyle='->', lw=2, color='black'))
ax.text(3, 6.3, 'Request', fontsize=8)

# Router -> Prefill
for i in range(3):
    y = 9.75 - i * 2.5
    ax.annotate('', xy=(7.5, y), xytext=(6, 6), 
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#e74c3c', connectionstyle='arc3,rad=0.1'))

# Prefill -> Decode (KV-cache transfer)
for i in range(3):
    y_from = 9.75 - i * 2.5
    y_to = 10.25 - i * 2
    ax.annotate('', xy=(13, y_to), xytext=(10, y_from),
                arrowprops=dict(arrowstyle='->', lw=2, color='#2ecc71', 
                               connectionstyle='arc3,rad=0.1', linestyle='--'))

ax.text(11.5, 10.5, 'KV-Cache\nTransfer', ha='center', fontsize=9, fontweight='bold', 
        color='#2ecc71', bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

# Response back
ax.annotate('', xy=(2.5, 5.5), xytext=(13, 3.5),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='gray', connectionstyle='arc3,rad=0.3'))
ax.text(7, 2.5, 'Streaming Response', fontsize=9, color='gray', fontweight='bold')

# Benefits box
benefits = (
    "Benefits:\n"
    "1. Independent scaling\n"
    "2. Better GPU utilization\n"
    "3. Lower TTFT (no decode blocking)\n"
    "4. Higher throughput (optimized batch sizes)"
)
ax.text(1, 2.5, benefits, fontsize=9, va='top',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='orange', alpha=0.9))

plt.tight_layout()
plt.show()

## 3. KV-Cache Transfer: The Critical Path

After prefill completes on a prefill node, the KV-cache must be transferred to a decode node. This transfer is the **critical path** that determines the feasibility of disaggregation.

### Transfer Size

For a single request with prompt length $s$:
$$\text{KV\_transfer\_size} = 2 \times n_{\text{layers}} \times n_{\text{KV\_heads}} \times d_{\text{head}} \times s \times \text{sizeof(dtype)}$$

In [ ]:
def kv_transfer_size(num_layers, num_kv_heads, head_dim, seq_len, dtype_bytes=2):
    """Calculate KV-cache transfer size in bytes."""
    return 2 * num_layers * num_kv_heads * head_dim * seq_len * dtype_bytes

def kv_transfer_time(size_bytes, bandwidth_gbps):
    """Transfer time in milliseconds."""
    return size_bytes / (bandwidth_gbps * 1e9) * 1000

# Model configs
models = {
    'Llama-3-8B': (32, 8, 128),
    'Llama-3-70B': (80, 8, 128),
    'Llama-3-405B': (126, 8, 128),
}

# Network options
networks = {
    'InfiniBand NDR (400Gb)': 50,   # GB/s
    'InfiniBand HDR (200Gb)': 25,   # GB/s
    'RoCE 100GbE': 12.5,            # GB/s
    'NVLink (intra-node)': 300,     # GB/s
}

seq_lens = [128, 512, 1024, 2048, 4096, 8192]

print("=== KV-Cache Transfer Analysis ===")
print(f"\n{'Model':<16} {'Seq Len':<10} {'Transfer Size':<15} ", end='')
for net in networks:
    print(f"{net[:12]:<14}", end='')
print()
print("-" * 110)

for model_name, (layers, kv_heads, head_dim) in models.items():
    for sl in [512, 2048, 8192]:
        size = kv_transfer_size(layers, kv_heads, head_dim, sl)
        size_str = f"{size/1024/1024:.1f} MB"
        print(f"{model_name:<16} {sl:<10} {size_str:<15}", end='')
        for net_name, bw in networks.items():
            t = kv_transfer_time(size, bw)
            print(f"{t:>8.1f}ms     ", end='')
        print()
    print()

# Visualize transfer time vs prefill time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Transfer size by model and sequence length
ax = axes[0]
x = np.arange(len(seq_lens))
width = 0.25
for i, (model_name, (layers, kv_heads, head_dim)) in enumerate(models.items()):
    sizes = [kv_transfer_size(layers, kv_heads, head_dim, sl) / 1024 / 1024 for sl in seq_lens]
    ax.bar(x + i*width - width, sizes, width, label=model_name, alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(seq_lens)
ax.set_xlabel('Sequence Length')
ax.set_ylabel('Transfer Size (MB)')
ax.set_title('KV-Cache Transfer Size', fontweight='bold')
ax.legend()
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3)

# Plot 2: Transfer time vs prefill time
ax = axes[1]
layers, kv_heads, head_dim = 80, 8, 128  # Llama-3-70B
for net_name, bw in [('IB NDR', 50), ('IB HDR', 25), ('100GbE', 12.5)]:
    transfer_times = [kv_transfer_time(kv_transfer_size(layers, kv_heads, head_dim, sl), bw) for sl in seq_lens]
    ax.plot(seq_lens, transfer_times, 'o-', label=f'Transfer ({net_name})', linewidth=2)

# Simulated prefill times
prefill_times = [s * 0.01 for s in seq_lens]  # ~0.01ms per token
ax.plot(seq_lens, prefill_times, 's--', color='#e74c3c', linewidth=2, label='Prefill time')

ax.set_xlabel('Sequence Length')
ax.set_ylabel('Time (ms)')
ax.set_title('Llama-3-70B: Transfer vs Prefill Time', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax.set_xscale('log', base=2)

plt.tight_layout()
plt.show()

## 4. Architecture Comparison: DistServe, Splitwise, Mooncake

Several systems have proposed disaggregated serving architectures:

### DistServe (2024)
- Separates prefill and decode into different GPU groups
- Uses a centralized scheduler that tracks both pools
- Transfers KV-cache via PCIe/NVLink for intra-node, IB for inter-node
- Focus: meeting SLO requirements for both TTFT and TPOT

### Splitwise (Microsoft, 2024)
- Similar disaggregation but within a single cluster
- Optimizes for mixed-phase execution within the same GPU cluster
- Uses token-level scheduling to interleave prefill and decode

### Mooncake (Moonshot AI, 2024)
- KV-cache stored in a distributed cache layer (disaggregated from both prefill and decode)
- Uses RDMA for fast KV-cache access
- Prefix caching is distributed across the cache pool
- Highly scalable: cache pool can scale independently

In [ ]:
# Compare architecture designs

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

def draw_box(ax, x, y, w, h, label, color, fontsize=8):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                                 facecolor=color, alpha=0.7, edgecolor='black'))
    ax.text(x+w/2, y+h/2, label, ha='center', va='center', fontsize=fontsize, fontweight='bold')

# === DistServe ===
ax = axes[0]
ax.set_title('DistServe', fontsize=13, fontweight='bold')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

draw_box(ax, 3, 8.5, 4, 1, 'Scheduler', '#f39c12')
draw_box(ax, 0.5, 5, 3.5, 2.5, 'Prefill Pool\n(GPU Group 1)', '#e74c3c')
draw_box(ax, 5.5, 5, 4, 2.5, 'Decode Pool\n(GPU Group 2)', '#3498db')
ax.annotate('', xy=(2.5, 7.5), xytext=(4, 8.5), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.annotate('', xy=(7, 7.5), xytext=(6, 8.5), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.annotate('', xy=(5.5, 6.25), xytext=(4, 6.25), 
            arrowprops=dict(arrowstyle='->', lw=2, color='#2ecc71'))
ax.text(4.75, 6.7, 'KV-cache', fontsize=7, ha='center', color='#2ecc71', fontweight='bold')
ax.text(5, 3.5, 'SLO-aware scheduling\nfor TTFT and TPOT', ha='center', fontsize=8, style='italic')

# === Splitwise ===
ax = axes[1]
ax.set_title('Splitwise', fontsize=13, fontweight='bold')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

draw_box(ax, 3, 8.5, 4, 1, 'Token-Level\nScheduler', '#f39c12')
# Mixed cluster
for i in range(2):
    for j in range(2):
        x = 1 + j * 4
        y = 4 + i * 2.5
        color = '#e74c3c' if (i+j) % 2 == 0 else '#3498db'
        role = 'Prefill' if (i+j) % 2 == 0 else 'Decode'
        draw_box(ax, x, y, 3.5, 1.5, f'GPU {i*2+j+1}\n({role})', color)

ax.text(5, 3, 'Same cluster,\nflexible role assignment', ha='center', fontsize=8, style='italic')

# === Mooncake ===
ax = axes[2]
ax.set_title('Mooncake', fontsize=13, fontweight='bold')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

draw_box(ax, 3, 8.5, 4, 1, 'Router', '#f39c12')
draw_box(ax, 0.5, 5, 3, 2, 'Prefill\nNodes', '#e74c3c')
draw_box(ax, 6.5, 5, 3, 2, 'Decode\nNodes', '#3498db')
draw_box(ax, 2.5, 1.5, 5, 2, 'Distributed KV-Cache Pool\n(RDMA-based)', '#2ecc71')

ax.annotate('', xy=(3, 3.5), xytext=(2, 5), arrowprops=dict(arrowstyle='<->', lw=1.5, color='#2ecc71'))
ax.annotate('', xy=(7, 3.5), xytext=(8, 5), arrowprops=dict(arrowstyle='<->', lw=1.5, color='#2ecc71'))
ax.text(5, 4.2, 'RDMA read/write', ha='center', fontsize=7, color='#2ecc71', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Simulation: Disaggregated vs Co-Located Serving

Let's build a simulation to compare disaggregated and co-located serving under the same workload.

In [ ]:
@dataclass
class Request:
    request_id: int
    arrival_time: float
    prompt_tokens: int
    gen_tokens: int
    
    # Timing results
    prefill_start: float = 0
    prefill_end: float = 0
    decode_start: float = 0
    decode_end: float = 0
    
    @property
    def ttft(self):
        return self.prefill_end - self.arrival_time
    
    @property
    def total_latency(self):
        return self.decode_end - self.arrival_time
    
    @property
    def tpot(self):
        if self.gen_tokens > 0:
            return (self.decode_end - self.decode_start) / self.gen_tokens
        return 0


class CoLocatedServer:
    """Simulates co-located serving (prefill and decode on same GPUs)."""
    
    def __init__(self, num_gpus: int = 4, prefill_speed: float = 10000,
                 decode_speed: float = 50, max_batch: int = 32):
        self.num_gpus = num_gpus
        self.prefill_speed = prefill_speed  # tokens/sec
        self.decode_speed = decode_speed  # tokens/sec per request
        self.max_batch = max_batch
    
    def simulate(self, requests: List[Request]) -> List[Request]:
        """Simulate co-located serving."""
        results = []
        current_time = 0
        queue = list(requests)
        active = []  # Requests currently decoding
        
        while queue or active:
            # Admit new requests (prefill)
            while queue and len(active) < self.max_batch:
                req = queue.pop(0)
                req.prefill_start = max(current_time, req.arrival_time)
                
                # Prefill blocks ALL decode during execution
                prefill_time = req.prompt_tokens / self.prefill_speed
                req.prefill_end = req.prefill_start + prefill_time
                req.decode_start = req.prefill_end
                
                # Advance time (prefill blocks decode!)
                current_time = req.prefill_end
                active.append(req)
                break  # Process one prefill per iteration
            
            if not active:
                if queue:
                    current_time = queue[0].arrival_time
                continue
            
            # Decode one token for all active requests
            decode_time = 1.0 / (self.decode_speed / len(active))  # Shared bandwidth
            current_time += decode_time
            
            completed = []
            for req in active:
                req.gen_tokens -= 1
                if req.gen_tokens <= 0:
                    req.decode_end = current_time
                    completed.append(req)
                    results.append(req)
            
            for req in completed:
                active.remove(req)
        
        return results


class DisaggregatedServer:
    """Simulates disaggregated serving (separate prefill and decode nodes)."""
    
    def __init__(self, num_prefill_gpus: int = 1, num_decode_gpus: int = 3,
                 prefill_speed: float = 10000, decode_speed: float = 50,
                 transfer_bandwidth_gbps: float = 50,
                 kv_bytes_per_token: int = 131072,  # Llama-3-70B
                 max_decode_batch: int = 64):
        self.num_prefill_gpus = num_prefill_gpus
        self.num_decode_gpus = num_decode_gpus
        self.prefill_speed = prefill_speed
        self.decode_speed = decode_speed
        self.transfer_bw = transfer_bandwidth_gbps
        self.kv_bytes_per_token = kv_bytes_per_token
        self.max_decode_batch = max_decode_batch
    
    def simulate(self, requests: List[Request]) -> List[Request]:
        """Simulate disaggregated serving."""
        results = []
        prefill_queue = list(requests)
        transfer_queue = []  # (ready_time, request)
        decode_active = []
        
        prefill_busy_until = 0
        current_time = 0
        
        for _ in range(100000):  # Max iterations
            # Find next event time
            next_times = []
            if prefill_queue:
                next_times.append(max(prefill_busy_until, prefill_queue[0].arrival_time))
            if transfer_queue:
                next_times.append(transfer_queue[0][0])
            if decode_active:
                next_times.append(current_time + 0.001)  # Decode step
            
            if not next_times:
                break
            
            current_time = min(next_times)
            
            # Process prefills (dedicated prefill node, doesn't block decode)
            if prefill_queue and current_time >= max(prefill_busy_until, prefill_queue[0].arrival_time):
                req = prefill_queue.pop(0)
                req.prefill_start = max(prefill_busy_until, req.arrival_time)
                prefill_time = req.prompt_tokens / self.prefill_speed
                req.prefill_end = req.prefill_start + prefill_time
                prefill_busy_until = req.prefill_end
                
                # KV-cache transfer
                transfer_size = req.prompt_tokens * self.kv_bytes_per_token
                transfer_time = transfer_size / (self.transfer_bw * 1e9)
                ready_time = req.prefill_end + transfer_time
                transfer_queue.append((ready_time, req))
                transfer_queue.sort(key=lambda x: x[0])
            
            # Check transfers completing
            while transfer_queue and transfer_queue[0][0] <= current_time:
                _, req = transfer_queue.pop(0)
                if len(decode_active) < self.max_decode_batch:
                    req.decode_start = current_time
                    decode_active.append(req)
            
            # Decode step (independent from prefill!)
            if decode_active:
                # Decode is memory-bound: throughput scales with batch
                effective_speed = self.decode_speed * min(len(decode_active), 8) / len(decode_active)
                
                completed = []
                for req in decode_active:
                    req.gen_tokens -= 1
                    if req.gen_tokens <= 0:
                        req.decode_end = current_time
                        completed.append(req)
                        results.append(req)
                
                for req in completed:
                    decode_active.remove(req)
            
            if not prefill_queue and not transfer_queue and not decode_active:
                break
        
        return results

# Generate workload
np.random.seed(42)
num_requests = 30
workload = []
arrival = 0
for i in range(num_requests):
    arrival += np.random.exponential(0.05)  # 20 req/sec
    workload.append(Request(
        request_id=i,
        arrival_time=arrival,
        prompt_tokens=int(np.random.exponential(300) + 50),
        gen_tokens=int(np.random.exponential(50) + 10),
    ))

# Run simulations
import copy

colocated = CoLocatedServer(num_gpus=4)
disagg = DisaggregatedServer(num_prefill_gpus=1, num_decode_gpus=3)

workload_co = [copy.deepcopy(r) for r in workload]
workload_da = [copy.deepcopy(r) for r in workload]

results_co = colocated.simulate(workload_co)
results_da = disagg.simulate(workload_da)

print(f"=== Simulation Results ({num_requests} requests) ===")
print(f"\n{'Metric':<25} {'Co-Located':<15} {'Disaggregated':<15} {'Improvement':<15}")
print("-" * 70)

metrics = [
    ('Avg TTFT (ms)', 
     np.mean([r.ttft * 1000 for r in results_co]),
     np.mean([r.ttft * 1000 for r in results_da])),
    ('P99 TTFT (ms)',
     np.percentile([r.ttft * 1000 for r in results_co], 99),
     np.percentile([r.ttft * 1000 for r in results_da], 99)),
    ('Avg Total Latency (ms)',
     np.mean([r.total_latency * 1000 for r in results_co]),
     np.mean([r.total_latency * 1000 for r in results_da])),
    ('Completed Requests',
     len(results_co), len(results_da)),
]

for name, co_val, da_val in metrics:
    improvement = (co_val - da_val) / co_val * 100 if co_val > 0 else 0
    print(f"{name:<25} {co_val:<15.1f} {da_val:<15.1f} {improvement:>+.1f}%")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: TTFT comparison
ax = axes[0, 0]
ttft_co = sorted([r.ttft * 1000 for r in results_co])
ttft_da = sorted([r.ttft * 1000 for r in results_da])
pcts = np.linspace(0, 100, len(ttft_co))
pcts_da = np.linspace(0, 100, len(ttft_da))

ax.plot(ttft_co, pcts, '-', color='#e74c3c', linewidth=2, label='Co-Located')
ax.plot(ttft_da, pcts_da, '-', color='#3498db', linewidth=2, label='Disaggregated')
ax.set_xlabel('TTFT (ms)')
ax.set_ylabel('Percentile')
ax.set_title('TTFT CDF', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Plot 2: Total latency
ax = axes[0, 1]
lat_co = [r.total_latency * 1000 for r in results_co]
lat_da = [r.total_latency * 1000 for r in results_da]
positions = [0, 1]
bp = ax.boxplot([lat_co, lat_da], positions=positions, widths=0.5,
                patch_artist=True)
bp['boxes'][0].set_facecolor('#e74c3c')
bp['boxes'][1].set_facecolor('#3498db')
ax.set_xticks(positions)
ax.set_xticklabels(['Co-Located', 'Disaggregated'])
ax.set_ylabel('Total Latency (ms)')
ax.set_title('Total Latency Distribution', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Plot 3: TTFT over time (shows prefill blocking)
ax = axes[1, 0]
arrivals_co = [r.arrival_time for r in results_co]
arrivals_da = [r.arrival_time for r in results_da]
ttfts_co = [r.ttft * 1000 for r in results_co]
ttfts_da = [r.ttft * 1000 for r in results_da]

ax.scatter(arrivals_co, ttfts_co, c='#e74c3c', alpha=0.6, label='Co-Located', s=30)
ax.scatter(arrivals_da, ttfts_da, c='#3498db', alpha=0.6, label='Disaggregated', s=30)
ax.set_xlabel('Arrival Time (s)')
ax.set_ylabel('TTFT (ms)')
ax.set_title('TTFT Over Time', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Plot 4: Summary bar chart
ax = axes[1, 1]
metrics_names = ['Avg TTFT\n(ms)', 'P99 TTFT\n(ms)', 'Avg Latency\n(ms)']
co_values = [
    np.mean([r.ttft * 1000 for r in results_co]),
    np.percentile([r.ttft * 1000 for r in results_co], 99),
    np.mean([r.total_latency * 1000 for r in results_co]),
]
da_values = [
    np.mean([r.ttft * 1000 for r in results_da]),
    np.percentile([r.ttft * 1000 for r in results_da], 99),
    np.mean([r.total_latency * 1000 for r in results_da]),
]

x = np.arange(len(metrics_names))
width = 0.35
ax.bar(x - width/2, co_values, width, label='Co-Located', color='#e74c3c', alpha=0.8)
ax.bar(x + width/2, da_values, width, label='Disaggregated', color='#3498db', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.set_title('Performance Comparison', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Co-Located vs Disaggregated Serving', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Scaling Analysis: When to Disaggregate

Disaggregation is not always beneficial. Let's analyze when it makes sense.

In [ ]:
def analyze_disaggregation_benefit(prompt_len, gen_len, arrival_rate,
                                     transfer_bw_gbps, kv_bytes_per_token):
    """Analyze whether disaggregation is beneficial."""
    # Prefill time
    prefill_time_ms = prompt_len * 0.01  # ~0.01ms per token
    
    # KV transfer time
    transfer_size = prompt_len * kv_bytes_per_token
    transfer_time_ms = transfer_size / (transfer_bw_gbps * 1e9) * 1000
    
    # Overhead ratio: transfer / prefill
    overhead_ratio = transfer_time_ms / prefill_time_ms if prefill_time_ms > 0 else float('inf')
    
    # Decode time (total)
    decode_time_ms = gen_len * 20  # ~20ms per token
    
    # Benefit: eliminate prefill-decode interference
    # Co-located: prefill blocks decode, causing TPOT spikes
    # Disaggregated: decode runs uninterrupted but has transfer overhead
    
    benefit_score = (
        (decode_time_ms / prefill_time_ms) *  # More decode = more benefit
        arrival_rate *  # Higher load = more interference
        (1 - overhead_ratio)  # Must offset transfer cost
    )
    
    return {
        'prefill_ms': prefill_time_ms,
        'transfer_ms': transfer_time_ms,
        'decode_ms': decode_time_ms,
        'overhead_ratio': overhead_ratio,
        'beneficial': overhead_ratio < 0.3 and arrival_rate > 5,
        'benefit_score': benefit_score,
    }

# Analyze across different workloads
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Transfer overhead vs prompt length
ax = axes[0]
prompt_lens = np.arange(64, 8193, 64)

for net_name, bw in [('IB NDR (400Gb)', 50), ('IB HDR (200Gb)', 25), ('100GbE', 12.5)]:
    overheads = []
    for pl in prompt_lens:
        result = analyze_disaggregation_benefit(pl, 100, 20, bw, 131072)
        overheads.append(result['overhead_ratio'] * 100)
    ax.plot(prompt_lens, overheads, '-', linewidth=2, label=net_name)

ax.axhline(y=30, color='red', linestyle='--', alpha=0.5, label='30% threshold')
ax.set_xlabel('Prompt Length (tokens)')
ax.set_ylabel('Transfer Overhead (%)')
ax.set_title('Transfer Overhead vs Prompt Length (Llama-3-70B)', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax.set_ylim(0, 100)

# Plot 2: When disaggregation is beneficial
ax = axes[1]
prompt_lens_2 = [128, 512, 2048, 4096]
arrival_rates = np.arange(1, 51)

for pl in prompt_lens_2:
    benefits = []
    for ar in arrival_rates:
        result = analyze_disaggregation_benefit(pl, 100, ar, 50, 131072)
        benefits.append(result['benefit_score'])
    ax.plot(arrival_rates, benefits, '-', linewidth=2, label=f'Prompt={pl}')

ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.set_xlabel('Arrival Rate (requests/sec)')
ax.set_ylabel('Disaggregation Benefit Score')
ax.set_title('When Is Disaggregation Beneficial?', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
ax.text(30, max(benefits)*0.8, 'Higher = more\nbenefit from\ndisaggregation', 
        fontsize=9, style='italic', color='green')

plt.tight_layout()
plt.show()

## 7. Network Bandwidth Requirements

The feasibility of disaggregation depends heavily on available network bandwidth.

In [ ]:
# Network bandwidth requirement analysis

def required_bandwidth(arrival_rate, avg_prompt_tokens, kv_bytes_per_token,
                        max_transfer_overhead_pct=20):
    """Calculate required network bandwidth for disaggregated serving."""
    # Total KV data per second
    kv_data_per_sec = arrival_rate * avg_prompt_tokens * kv_bytes_per_token
    
    # Need to transfer within prefill_time * overhead_pct
    prefill_time_per_req = avg_prompt_tokens * 0.00001  # seconds
    max_transfer_time = prefill_time_per_req * (max_transfer_overhead_pct / 100)
    
    # Bandwidth needed per request
    data_per_req = avg_prompt_tokens * kv_bytes_per_token
    bw_per_req = data_per_req / max_transfer_time
    
    # Total bandwidth with concurrent transfers
    # Assume pipelining allows overlap
    total_bw_needed = kv_data_per_sec  # Sustained bandwidth
    
    return {
        'data_per_request_mb': data_per_req / 1024 / 1024,
        'sustained_bw_gbps': total_bw_needed / 1e9,
        'peak_bw_gbps': bw_per_req / 1e9,
    }

print("=== Network Bandwidth Requirements ===")
print(f"\n{'Scenario':<35} {'Data/Req':<12} {'Sustained BW':<15} {'Peak BW':<12}")
print("-" * 75)

scenarios = [
    ('Llama-3-8B, 10 req/s, 512 tok', 10, 512, 65536),
    ('Llama-3-70B, 10 req/s, 512 tok', 10, 512, 131072),
    ('Llama-3-70B, 20 req/s, 1024 tok', 20, 1024, 131072),
    ('Llama-3-70B, 50 req/s, 2048 tok', 50, 2048, 131072),
    ('Llama-3-405B, 10 req/s, 512 tok', 10, 512, 258048),
]

for name, rate, tokens, kv_bytes in scenarios:
    result = required_bandwidth(rate, tokens, kv_bytes)
    print(f"{name:<35} {result['data_per_request_mb']:<10.1f}MB "
          f"{result['sustained_bw_gbps']:<13.1f}GB/s "
          f"{result['peak_bw_gbps']:<10.1f}GB/s")

## 8. Prefill/Decode Ratio Optimization

A key design decision is how many GPUs to allocate to prefill vs decode.

In [ ]:
def optimize_ratio(total_gpus: int, arrival_rate: float,
                    avg_prompt_tokens: int, avg_gen_tokens: int):
    """Find optimal prefill:decode GPU ratio."""
    results = []
    
    for num_prefill in range(1, total_gpus):
        num_decode = total_gpus - num_prefill
        
        # Prefill throughput: tokens/sec per GPU
        prefill_tps = 10000  # tokens/sec
        total_prefill_capacity = num_prefill * prefill_tps / avg_prompt_tokens  # requests/sec
        
        # Decode throughput: depends on batch size
        # Each decode GPU can handle ~50 concurrent requests
        max_concurrent = num_decode * 50
        avg_decode_time = avg_gen_tokens * 0.02  # 20ms per token
        decode_capacity = max_concurrent / avg_decode_time  # requests/sec
        
        # System throughput limited by bottleneck
        system_throughput = min(total_prefill_capacity, decode_capacity)
        
        # Can we handle the arrival rate?
        can_handle = system_throughput >= arrival_rate
        
        # Utilization
        prefill_util = min(1.0, arrival_rate / total_prefill_capacity)
        decode_util = min(1.0, arrival_rate / decode_capacity) if decode_capacity > 0 else 1.0
        balance = 1.0 - abs(prefill_util - decode_util)  # 1.0 = perfectly balanced
        
        results.append({
            'prefill_gpus': num_prefill,
            'decode_gpus': num_decode,
            'ratio': f"{num_prefill}:{num_decode}",
            'prefill_capacity': total_prefill_capacity,
            'decode_capacity': decode_capacity,
            'system_throughput': system_throughput,
            'can_handle': can_handle,
            'prefill_util': prefill_util,
            'decode_util': decode_util,
            'balance': balance,
        })
    
    return results

# Optimize for 8 GPUs
results = optimize_ratio(8, arrival_rate=20, avg_prompt_tokens=500, avg_gen_tokens=100)

print("=== GPU Ratio Optimization (8 GPUs total, 20 req/s) ===")
print(f"{'Ratio':<8} {'Prefill Cap':<14} {'Decode Cap':<13} {'Throughput':<12} "
      f"{'P-Util':<8} {'D-Util':<8} {'Balance':<8} {'OK?':<5}")
print("-" * 80)

best = max(results, key=lambda r: r['balance'] if r['can_handle'] else -1)

for r in results:
    marker = ' *' if r == best else ''
    print(f"{r['ratio']:<8} {r['prefill_capacity']:<14.1f} {r['decode_capacity']:<13.1f} "
          f"{r['system_throughput']:<12.1f} {r['prefill_util']:<8.1%} {r['decode_util']:<8.1%} "
          f"{r['balance']:<8.2f} {'Yes' if r['can_handle'] else 'No':<5}{marker}")

print(f"\nOptimal ratio: {best['ratio']} (balance={best['balance']:.2f})")

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
ratios = [r['ratio'] for r in results]
prefill_utils = [r['prefill_util'] * 100 for r in results]
decode_utils = [r['decode_util'] * 100 for r in results]
balances = [r['balance'] * 100 for r in results]

x = np.arange(len(ratios))
width = 0.25
ax.bar(x - width, prefill_utils, width, label='Prefill Utilization', color='#e74c3c', alpha=0.7)
ax.bar(x, decode_utils, width, label='Decode Utilization', color='#3498db', alpha=0.7)
ax.bar(x + width, balances, width, label='Balance Score', color='#2ecc71', alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels(ratios)
ax.set_xlabel('Prefill:Decode GPU Ratio')
ax.set_ylabel('Percentage')
ax.set_title('Finding Optimal Prefill:Decode GPU Ratio', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Exercises

### Exercise 1: Design a Disaggregated Architecture

In [ ]:
# Exercise 1: Design a disaggregated architecture

"""
YOUR TASK: Design a disaggregated serving architecture for the following workload:

Model: Llama-3-70B
- 80 layers, 8192 hidden dim, 8 KV heads, 128 head dim
- FP16 weights: 140GB

Workload:
- Average arrival rate: 30 requests/sec
- Average prompt length: 1024 tokens
- Average generation length: 200 tokens
- P99 TTFT target: 500ms
- P99 TPOT target: 50ms

Hardware:
- Available: 16x H100 GPUs (80GB each)
- Intra-node: NVLink (450 GB/s)
- Inter-node: InfiniBand NDR (400 Gb/s = 50 GB/s)
- 2 nodes, 8 GPUs per node

Design decisions:
1. How many prefill vs decode GPUs?
2. Which GPUs go in which role?
3. TP size for each role?
4. Estimated KV transfer time?
5. Can you meet the SLO targets?
"""

# YOUR ANALYSIS HERE
# Start by calculating:
kv_per_token = 2 * 80 * 8 * 128 * 2  # bytes
kv_for_1024 = kv_per_token * 1024
print(f"KV-cache per token: {kv_per_token/1024:.1f} KB")
print(f"KV-cache for 1024 tokens: {kv_for_1024/1024/1024:.2f} GB")
print(f"Transfer time (IB NDR): {kv_for_1024 / (50 * 1e9) * 1000:.1f} ms")

print("\nYOUR TASK: Complete the architecture design.")
print("Consider: TP requirements, memory per GPU, prefill/decode ratio.")

### Exercise 2: Implement KV-Cache Transfer Optimization

In [ ]:
# Exercise 2: KV-Cache transfer optimization

class KVTransferOptimizer:
    """YOUR CODE: Implement strategies to reduce KV-cache transfer overhead.
    
    Strategies to implement:
    1. Pipelined transfer: Start transferring early layers while later layers compute
    2. Compression: Quantize KV-cache to INT8 before transfer (2x reduction)
    3. Selective transfer: For GQA models, only transfer unique KV heads
    4. Overlap with decode: Begin decode on partially transferred cache
    """
    
    def pipelined_transfer_time(self, num_layers, kv_per_layer_bytes, 
                                  bandwidth_gbps, prefill_time_per_layer_ms):
        """YOUR CODE: Calculate transfer time with layer pipelining."""
        pass
    
    def compressed_transfer_time(self, total_bytes, bandwidth_gbps, 
                                   compression_ratio=2.0):
        """YOUR CODE: Calculate transfer time with compression."""
        pass

print("Exercise 2: Implement KVTransferOptimizer.")
print("Compare transfer times with and without each optimization.")

### Exercise 3: Auto-Scaling Simulation

In [ ]:
# Exercise 3: Auto-scaling for disaggregated serving

class AutoScaler:
    """YOUR CODE: Implement auto-scaling for disaggregated serving.
    
    Monitor:
    1. Prefill queue depth
    2. Decode batch utilization
    3. TTFT and TPOT metrics
    
    Actions:
    1. Scale up prefill GPUs if queue is growing
    2. Scale up decode GPUs if batch utilization is high
    3. Scale down if utilization is low for sustained period
    4. Rebalance if one pool is overprovisioned
    """
    
    def __init__(self, min_prefill=1, max_prefill=8, min_decode=1, max_decode=8):
        self.min_prefill = min_prefill
        self.max_prefill = max_prefill
        self.min_decode = min_decode
        self.max_decode = max_decode
    
    def evaluate(self, metrics: dict) -> dict:
        """YOUR CODE: Decide scaling actions based on metrics."""
        pass

print("Exercise 3: Implement AutoScaler.")
print("Simulate a workload with varying arrival rates and verify scaling decisions.")

## 10. Summary

In this chapter, we explored disaggregated LLM serving -- a cutting-edge architecture that separates prefill and decode:

1. **The Mismatch**: Prefill is compute-bound; decode is memory-bandwidth-bound. Co-locating them wastes resources.

2. **Architecture**: Separate prefill and decode pools connected by high-bandwidth networking, with KV-cache transfer as the critical path.

3. **KV-Cache Transfer**: For Llama-3-70B with 1K tokens, ~131MB must be transferred per request. IB NDR handles this in ~2.6ms.

4. **Systems**:
   - **DistServe**: SLO-aware scheduling with separate pools
   - **Splitwise**: Flexible role assignment within a cluster
   - **Mooncake**: Fully disaggregated with shared KV-cache pool

5. **Benefits**:
   - Independent scaling of prefill and decode
   - Better GPU utilization (each pool optimized for its workload)
   - Lower TTFT (no decode blocking during prefill)
   - Higher throughput (decode can use larger batch sizes)

6. **When to Disaggregate**:
   - High arrival rate (>10 req/s)
   - Moderate-to-long prompts (>256 tokens)
   - Sufficient network bandwidth (IB NDR or better)
   - When SLO requirements for both TTFT and TPOT are tight

### Key Takeaway

Disaggregation is the logical next step in LLM serving optimization. By recognizing that prefill and decode are fundamentally different workloads, we can design systems that efficiently use GPU resources for both.

## What's Next?

This concludes Part 2: LLM Serving System Design. In Part 3, we'll dive deep into the **attention mechanism optimizations** that make efficient serving possible, including FlashAttention, PagedAttention, and RadixAttention.